In [0]:
# Silver Layer - Clean, join all 3 bronze tables
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ── Read Bronze Tables ────────────────────────────────────
fmcg_df = spark.read.format("delta").table("retail_shelf_intelligence.raw_data.bronze_fmcg_sales")
weather_df = spark.read.format("delta").table("retail_shelf_intelligence.raw_data.bronze_weather")
holidays_df = spark.read.format("delta").table("retail_shelf_intelligence.raw_data.bronze_holidays")

# ── Clean FMCG Data ───────────────────────────────────────
fmcg_clean = fmcg_df \
    .dropDuplicates(["Invoice_ID"]) \
    .dropna(subset=["Invoice_ID", "Invoice_Date", "City", "Stock_On_Hand"]) \
    .withColumn("Invoice_Date", to_date(col("Invoice_Date"))) \
    .withColumn("City", trim(col("City"))) \
    .withColumn("Stock_Risk_Flag", 
        when(col("Stock_On_Hand") <= col("Reorder_Level"), 1).otherwise(0))

# ── Clean Weather Data ────────────────────────────────────
weather_clean = weather_df \
    .dropna() \
    .withColumn("date", to_date(col("date"))) \
    .withColumnRenamed("city", "weather_city") \
    .drop("ingested_at")

# ── Clean Holidays Data ───────────────────────────────────
holidays_clean = holidays_df \
    .withColumn("date", to_date(col("date"))) \
    .drop("ingested_at")

# ── Join All 3 Sources ────────────────────────────────────
silver_df = fmcg_clean \
    .join(weather_clean, 
        (fmcg_clean.Invoice_Date == weather_clean.date) & 
        (fmcg_clean.City == weather_clean.weather_city), 
        "left") \
    .join(holidays_clean,
        fmcg_clean.Invoice_Date == holidays_clean.date,
        "left") \
    .withColumn("is_holiday", 
        when(col("holiday_name").isNotNull(), 1).otherwise(0)) \
    .drop("date", "weather_city", "ingested_at")

# ── Save Silver Table ─────────────────────────────────────
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_shelf_intelligence.raw_data.silver_retail")

print(f"Silver layer saved: {silver_df.count()} rows")
print("\n✅ Silver layer complete!")